***Installing dependencies***

In [ ]:
# Colab: نصب کتابخانه‌های مورد نیاز
!pip install -q transformers datasets accelerate peft evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


***report***

In [ ]:
import torch
import os
import time

def report_model_info(model, trainer, tag="Full Fine-tune", extra_info=None):
    print("="*80)
    print(f"📊 گزارش برای: {tag}")
    print("="*80)

    # معماری مدل
    cfg = model.config
    print("🔹 معماری مدل:")
    print(f"  - num_hidden_layers: {cfg.num_hidden_layers}")
    print(f"  - hidden_size: {cfg.hidden_size}")
    print(f"  - num_attention_heads: {cfg.num_attention_heads}")
    print()

    # هایپرپارامترها
    print("🔹 هایپرپارامترهای آموزش:")
    print(f"  - batch_size: {trainer.args.per_device_train_batch_size}")
    print(f"  - num_train_epochs: {trainer.args.num_train_epochs}")
    print(f"  - learning_rate: {trainer.args.learning_rate}")
    if extra_info:
        for k,v in extra_info.items():
            print(f"  - {k}: {v}")
    print()

    # تعداد پارامترهای آموزش‌پذیر
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print("🔹 پارامترها:")
    print(f"  - trainable parameters: {trainable:,}")
    print(f"  - total parameters: {total:,}")
    print(f"  - درصد پارامترهای آموزش‌پذیر: {100*trainable/total:.2f}%")
    print()

    # ارزیابی روی validation
    print("🔹 ارزیابی روی validation:")
    eval_metrics = trainer.evaluate()
    print(eval_metrics)
    print()

    # حجم فایل ذخیره‌شده
    ckpt_dir = trainer.args.output_dir
    if os.path.exists(ckpt_dir):
        size_mb = sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, fn in os.walk(ckpt_dir) for f in fn) / 1e6
        print(f"🔹 حجم پوشه‌ی مدل ذخیره‌شده: {size_mb:.2f} MB")
    else:
        print("⚠️ پوشه‌ی checkpoint پیدا نشد.")
    print("="*80, "\n")


***Loading libraries and initialization***

In [ ]:
import os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate
import numpy as np

# PEFT
from peft import LoraConfig, get_peft_model, TaskType, PromptTuningConfig, get_peft_model  # prompt tuning config هم داریم

# انتخاب مدل پایه
MODEL_NAME = "roberta-large"  # یا "FacebookAI/roberta-large"
NUM_LABELS = 3  # MNLI: entailment / neutral / contradiction

# متریک
accuracy = evaluate.load("accuracy")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


***Dataset loading (MultiNLI) and 10% sampling***

In [ ]:
# بارگذاری MultiNLI از Hugging Face datasets
raw_datasets = load_dataset("multi_nli")

# برای صرفه‌جویی، فقط 10% از train را انتخاب می‌کنیم
train10pct = raw_datasets["train"].train_test_split(test_size=0.90, shuffle=True, seed=42)["train"]
# validation و test (dev_matched) را نگه‌داریم
validation = raw_datasets["validation_matched"]

print("train size (10%):", len(train10pct))
print("validation size:", len(validation))
print("\n📊 ساختار dataset:")
print(raw_datasets)

print("\n🔍 کلیدهای موجود:")
print(raw_datasets.keys())

# نمایش اطلاعات هر split
for split_name in raw_datasets.keys():
    print(f"\n📈 اطلاعات split '{split_name}':")
    print(f"   تعداد نمونه‌ها: {len(raw_datasets[split_name])}")
    print(f"   ویژگی‌ها: {raw_datasets[split_name].features}")

# نمایش 5 نمونه از داده آموزش (train)
print("\n" + "="*50)
print("🎯 5 نمونه از داده‌های آموزش (train):")
print("="*50)

train_df = raw_datasets['train'].to_pandas()
print(train_df.head(5))

# نمایش 5 نمونه از داده validation
print("\n" + "="*50)
print("🔍 5 نمونه از داده‌های validation (validation_matched):")
print("="*50)

validation_df = raw_datasets['validation_matched'].to_pandas()
print(validation_df.head(5))

# نمایش یک نمونه به صورت کامل با جزئیات
print("\n" + "="*50)
print("🔎 یک نمونه کامل با جزئیات:")
print("="*50)

sample = raw_datasets['train'][0]
for key, value in sample.items():
    print(f"🔑 {key}:")
    print(f"   {value}")
    print("-" * 30)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/214M [00:00<?, ?B/s]

data/validation_matched-00000-of-00001.p(…):   0%|          | 0.00/4.94M [00:00<?, ?B/s]

data/validation_mismatched-00000-of-0000(…):   0%|          | 0.00/5.10M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

train size (10%): 39270
validation size: 9815

📊 ساختار dataset:
DatasetDict({
    train: Dataset({
        features: ['promptID', 'pairID', 'premise', 'premise_binary_parse', 'premise_parse', 'hypothesis', 'hypothesis_binary_parse', 'hypothesis_parse', 'genre', 'label'],
        num_rows: 392702
    })
    validation_matched: Dataset({
        features: ['promptID', 'pairID', 'premise', 'premise_binary_parse', 'premise_parse', 'hypothesis', 'hypothesis_binary_parse', 'hypothesis_parse', 'genre', 'label'],
        num_rows: 9815
    })
    validation_mismatched: Dataset({
        features: ['promptID', 'pairID', 'premise', 'premise_binary_parse', 'premise_parse', 'hypothesis', 'hypothesis_binary_parse', 'hypothesis_parse', 'genre', 'label'],
        num_rows: 9832
    })
})

🔍 کلیدهای موجود:
dict_keys(['train', 'validation_matched', 'validation_mismatched'])

📈 اطلاعات split 'train':
   تعداد نمونه‌ها: 392702
   ویژگی‌ها: {'promptID': Value('int32'), 'pairID': Value('string'), 'premise

***Tokenizer and preprocessing***

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

# 1️⃣ تعریف مدل / توکنایزر
MODEL_NAME = "roberta-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2️⃣ تابع پردازش
def preprocess_function(examples):
    # premise = sentence1, hypothesis = sentence2
    return tokenizer(examples["premise"], examples["hypothesis"], truncation=True)

# 3️⃣ توکنایز کردن دیتاست
# ستون 'label' را نگه می‌داریم
encoded_train = train10pct.map(
    preprocess_function,
    batched=True,
    remove_columns=[c for c in train10pct.column_names if c not in ["label"]]  # فقط ستون label نگه داشته میشه
)
encoded_val = validation.map(
    preprocess_function,
    batched=True,
    remove_columns=[c for c in validation.column_names if c not in ["label"]]
)

# 4️⃣ rename ستون label به labels (مورد نیاز Trainer)
encoded_train = encoded_train.rename_column("label", "labels")
encoded_val   = encoded_val.rename_column("label", "labels")

# 5️⃣ فرمت PyTorch
encoded_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
encoded_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# 6️⃣ data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)



tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/39270 [00:00<?, ? examples/s]

Map:   0%|          | 0/9815 [00:00<?, ? examples/s]

***compute_metrics function for Trainer***

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)


***Scenario A — Full fine-tune (all parameters)***

***Preparing the model for classification (update all parameters)***

In [ ]:
model_full = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


***TrainingArguments and Trainer (Full)***

In [ ]:
from transformers import RobertaForSequenceClassification, TrainingArguments, Trainer

# 1️⃣ مدل برای classification
model_full = RobertaForSequenceClassification.from_pretrained(
    "roberta-large",
    num_labels=3
)

# 2️⃣ اطمینان از وجود ستون labels در dataset
# اگر ستون هنوز label هست، rename کن
if "label" in encoded_train.column_names:
    encoded_train = encoded_train.rename_column("label", "labels")
if "label" in encoded_val.column_names:
    encoded_val = encoded_val.rename_column("label", "labels")

# فرمت PyTorch
encoded_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
encoded_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# 3️⃣ تعداد استپ‌ها در هر epoch
steps_per_epoch = len(encoded_train) // 4   # batch_size=4

# 4️⃣ آرگومان‌های آموزش
training_args_full = TrainingArguments(
    output_dir="./results/full_finetune",
    do_eval=True,                   # فعال کردن ارزیابی
    eval_steps=steps_per_epoch,      # معادل ارزیابی در پایان هر epoch
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_total_limit=1,
    fp16=True,
    logging_steps=50,
    push_to_hub=False,
)

# 5️⃣ متریک accuracy
import evaluate
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 6️⃣ تعریف Trainer
trainer_full = Trainer(
    model=model_full,
    args=training_args_full,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 7️⃣ آموزش
trainer_full.train()

# 8️⃣ گزارش
report_model_info(model_full, trainer_full, tag="Full Fine-tune")



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-688901609.py:49: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_full = Trainer(


Step,Training Loss
50,1.273600
100,1.119400
150,1.169900
200,1.173200
250,1.128700
300,1.128200
350,1.116200
400,1.117400
450,1.147900
500,1.130400


Step,Training Loss
50,1.273600
100,1.119400
150,1.169900
200,1.173200
250,1.128700
300,1.128200
350,1.116200
400,1.117400
450,1.147900
500,1.130400


KeyboardInterrupt: 

***Scenario B — LoRA (PEFT) — Low-volume parameters\***

***Loading the LoRA model and configuring it***

In [ ]:
model_lora = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

# پیکربندی LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,   # چون داریم classification انجام میدیم
    inference_mode=False,
    r=8,           # rank کم (می‌تونی تغییر بدی)
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    # target_modules می‌تونه بسته به نام لایه‌های مدل تغییر کنه؛
    # برای RoBERTa معمولاً projection های attention را هدف قرار می‌دهیم.
    target_modules=["query", "value"]
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()  # نمایش درصد/تعداد پارامترهای آموزشی


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,839,107 || all params: 357,201,926 || trainable%: 0.5149


***TrainingArguments and Trainer for LoRA***

In [ ]:
from transformers import TrainingArguments, Trainer

# تعداد استپ‌ها در هر epoch
steps_per_epoch = len(encoded_train) // 8  # batch_size=8

training_args_lora = TrainingArguments(
    output_dir="./results/lora_finetune",
    do_eval=True,                   # فعال کردن ارزیابی
    eval_steps=steps_per_epoch,      # معادل ارزیابی در پایان هر epoch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=3e-4,  # معمولاً lr برای LoRA بالاتر است
    weight_decay=0.0,
    save_total_limit=1,
    fp16=True,
    logging_steps=50,
    push_to_hub=False,
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer_lora.train()
report_model_info(model_lora, trainer_lora, tag="LoRA", extra_info={"lora_r": 8})


/tmp/ipython-input-3710507592.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_lora = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: f-saghafi75 (f-saghafi75-university-of-tehran) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
50,1.166100
100,1.162600
150,1.129800
200,1.150600
250,1.171700
300,1.134500
350,1.139800
400,1.113500
450,1.129700
500,1.128800


Step,Training Loss
50,1.166100
100,1.162600
150,1.129800
200,1.150600
250,1.171700
300,1.134500
350,1.139800
400,1.113500
450,1.129700
500,1.128800


📊 گزارش برای: LoRA
🔹 معماری مدل:
  - num_hidden_layers: 24
  - hidden_size: 1024
  - num_attention_heads: 16

🔹 هایپرپارامترهای آموزش:
  - batch_size: 8
  - num_train_epochs: 3
  - learning_rate: 0.0003
  - lora_r: 8

🔹 پارامترها:
  - trainable parameters: 1,839,107
  - total parameters: 357,201,926
  - درصد پارامترهای آموزش‌پذیر: 0.51%

🔹 ارزیابی روی validation:


{'eval_loss': 1.0991214513778687, 'eval_accuracy': 0.31818644931227713, 'eval_runtime': 37.9161, 'eval_samples_per_second': 258.861, 'eval_steps_per_second': 16.194, 'epoch': 3.0}

🔹 حجم پوشه‌ی مدل ذخیره‌شده: 27.13 MB



 ***Scenario C—Prompt Tuning (Soft-Prompt / Tuning-P) with PEFT***

***Configure PromptTuning and model wrap***

In [ ]:
model_prompt = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

prompt_config = PromptTuningConfig(
    task_type="SEQ_CLS",
    num_virtual_tokens=50,  # تعداد توکن‌هایی که به‌عنوان soft prompt می‌سازیم
    token_dim=model_prompt.config.hidden_size
)

# PEFT API: برخی نسخه‌ها PromptTuningConfig را متفاوت می‌پذیرند؛ اگر خطا دیدی مستندات PEFT را برای نسخه‌ات بررسی کن.
model_prompt = get_peft_model(model_prompt, prompt_config)
model_prompt.print_trainable_parameters()


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,103,875 || all params: 356,466,694 || trainable%: 0.3097


***TrainingArguments and Trainer for PromptTuning***

In [ ]:
from transformers import TrainingArguments, Trainer

# تعداد استپ‌ها در هر epoch
steps_per_epoch = len(encoded_train) // 16  # batch_size=16

training_args_prompt = TrainingArguments(
    output_dir="./results/prompt_tuning",
    do_eval=True,                   # فعال کردن ارزیابی
    eval_steps=steps_per_epoch,      # معادل ارزیابی در پایان هر epoch
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,  # prompt tuning ممکنه نیاز به epochs بیشتری داشته باشه
    learning_rate=5e-4,
    weight_decay=0.0,
    save_total_limit=1,
    fp16=True,
    logging_steps=50,
    push_to_hub=False,
)

trainer_prompt = Trainer(
    model=model_prompt,
    args=training_args_prompt,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer_prompt.train()

report_model_info(model_prompt, trainer_prompt, tag="Prompt Tuning", extra_info={"num_virtual_tokens": 50})


/tmp/ipython-input-1497233502.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_prompt = Trainer(


Step,Training Loss
50,1.071600
100,1.058200
150,1.120300
200,1.175800
250,1.122600
300,1.168900
350,1.158600
400,1.120800
450,1.161000
500,1.125600


📊 گزارش برای: Prompt Tuning
🔹 معماری مدل:
  - num_hidden_layers: 24
  - hidden_size: 1024
  - num_attention_heads: 16

🔹 هایپرپارامترهای آموزش:
  - batch_size: 16
  - num_train_epochs: 3
  - learning_rate: 0.0005
  - num_virtual_tokens: 50

🔹 پارامترها:
  - trainable parameters: 1,103,875
  - total parameters: 356,466,694
  - درصد پارامترهای آموزش‌پذیر: 0.31%

🔹 ارزیابی روی validation:


{'eval_loss': 1.0645828247070312, 'eval_accuracy': 0.41803362200713196, 'eval_runtime': 55.2365, 'eval_samples_per_second': 177.69, 'eval_steps_per_second': 5.558, 'epoch': 3.0}

🔹 حجم پوشه‌ی مدل ذخیره‌شده: 18.17 MB

